# Chapter 23: Integrated Case Studies — Full Offshore Platform Simulation

This chapter brings together the techniques from previous chapters into a complete
offshore platform process simulation. We build a 3-stage separation train (HP, MP, LP),
followed by gas recompression to export pressure, and analyze production rates,
equipment utilization, and power consumption.

**Key learning objectives:**
- Build an integrated multi-stage separation and compression process
- Extract and analyze production rates for oil and gas
- Evaluate equipment performance and power consumption
- Visualize process conditions across the entire plant

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 23.1 Define the Well Fluid

We define a typical North Sea well fluid with light and intermediate hydrocarbons,
plus CO2 and water. The wellhead conditions are 80 bara and 80 °C.

In [2]:
# Define well fluid composition (mole fractions)
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 80.0, 80.0)
fluid.addComponent("nitrogen", 0.005)
fluid.addComponent("CO2", 0.025)
fluid.addComponent("methane", 0.60)
fluid.addComponent("ethane", 0.08)
fluid.addComponent("propane", 0.05)
fluid.addComponent("i-butane", 0.015)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("i-pentane", 0.01)
fluid.addComponent("n-pentane", 0.01)
fluid.addComponent("n-hexane", 0.02)
fluid.addComponent("n-heptane", 0.03)
fluid.addComponent("n-octane", 0.025)
fluid.addComponent("n-nonane", 0.01)
fluid.addComponent("water", 0.10)
fluid.setMixingRule("classic")
fluid.setMultiPhaseCheck(True)

print(f"Number of components: {fluid.getNumberOfComponents()}")
print(f"Feed temperature: {fluid.getTemperature() - 273.15:.1f} C")
print(f"Feed pressure: {fluid.getPressure():.1f} bara")

Number of components: 14
Feed temperature: 80.0 C
Feed pressure: 80.0 bara


## 23.2 Build the 3-Stage Separation and Compression Train

The process consists of:
1. **HP Separator** at 80 bara — first-stage gas removal
2. **MP Separator** at 20 bara — intermediate flash
3. **LP Separator** at 3 bara — final oil stabilization
4. **LP Compressor** — recompress LP gas to MP pressure
5. **Export Compressor** — compress combined gas to export pressure (150 bara)

In [3]:
from neqsim import jneqsim

# Create the well stream
well_stream = jneqsim.process.equipment.stream.Stream("Well Stream", fluid)
well_stream.setFlowRate(50000.0, "kg/hr")
well_stream.setTemperature(80.0, "C")
well_stream.setPressure(80.0, "bara")

# HP Separator (80 bara)
hp_sep = jneqsim.process.equipment.separator.ThreePhaseSeparator("HP Separator", well_stream)

# Throttle HP oil to MP pressure
hp_oil_valve = jneqsim.process.equipment.valve.ThrottlingValve("HP-MP Valve", hp_sep.getOilOutStream())
hp_oil_valve.setOutletPressure(20.0)

# MP Separator (20 bara)
mp_sep = jneqsim.process.equipment.separator.ThreePhaseSeparator("MP Separator", hp_oil_valve.getOutletStream())

# Throttle MP oil to LP pressure
mp_oil_valve = jneqsim.process.equipment.valve.ThrottlingValve("MP-LP Valve", mp_sep.getOilOutStream())
mp_oil_valve.setOutletPressure(3.0)

# LP Separator (3 bara)
lp_sep = jneqsim.process.equipment.separator.ThreePhaseSeparator("LP Separator", mp_oil_valve.getOutletStream())

# LP Compressor: compress LP gas from 3 bara to 20 bara
lp_compressor = jneqsim.process.equipment.compressor.Compressor("LP Compressor", lp_sep.getGasOutStream())
lp_compressor.setOutletPressure(20.0)
lp_compressor.setPolytropicEfficiency(0.75)

# After-cooler for LP compressor discharge
lp_cooler = jneqsim.process.equipment.heatexchanger.Cooler("LP After-Cooler", lp_compressor.getOutletStream())
lp_cooler.setOutTemperature(273.15 + 35.0)

# Mix MP gas and compressed LP gas
gas_mixer = jneqsim.process.equipment.mixer.Mixer("Gas Mixer")
gas_mixer.addStream(hp_sep.getGasOutStream())
gas_mixer.addStream(mp_sep.getGasOutStream())
gas_mixer.addStream(lp_cooler.getOutletStream())

# Export Compressor: compress combined gas to 150 bara
export_compressor = jneqsim.process.equipment.compressor.Compressor("Export Compressor", gas_mixer.getOutletStream())
export_compressor.setOutletPressure(150.0)
export_compressor.setPolytropicEfficiency(0.78)

# Export cooler
export_cooler = jneqsim.process.equipment.heatexchanger.Cooler("Export Cooler", export_compressor.getOutletStream())
export_cooler.setOutTemperature(273.15 + 40.0)

# Build and run the process
process = jneqsim.process.processmodel.ProcessSystem()
process.add(well_stream)
process.add(hp_sep)
process.add(hp_oil_valve)
process.add(mp_sep)
process.add(mp_oil_valve)
process.add(lp_sep)
process.add(lp_compressor)
process.add(lp_cooler)
process.add(gas_mixer)
process.add(export_compressor)
process.add(export_cooler)
process.run()

print("Process simulation complete.")

Process simulation complete.


## 23.3 Production Rates and Equipment Summary

Extract key production metrics: gas export rate, oil export rate, produced water rate,
and compressor power consumption.

In [4]:
# Gas export
gas_export = export_cooler.getOutletStream()
gas_rate_MSm3d = gas_export.getFlowRate("MSm3/day")
gas_rate_kg_hr = gas_export.getFlowRate("kg/hr")

# Oil export from LP separator
oil_export = lp_sep.getOilOutStream()
oil_rate_m3_hr = oil_export.getFlowRate("m3/hr")
oil_rate_kg_hr = oil_export.getFlowRate("kg/hr")

# Water from separators
water_hp = hp_sep.getWaterOutStream().getFlowRate("m3/hr")
water_mp = mp_sep.getWaterOutStream().getFlowRate("m3/hr")
water_lp = lp_sep.getWaterOutStream().getFlowRate("m3/hr")
total_water = water_hp + water_mp + water_lp

# Compressor power
lp_power_kW = lp_compressor.getPower() / 1000.0  # W to kW
export_power_kW = export_compressor.getPower() / 1000.0
total_power_kW = lp_power_kW + export_power_kW

print("=" * 60)
print("PRODUCTION SUMMARY")
print("=" * 60)
print(f"Gas export rate:       {gas_rate_MSm3d:.4f} MSm3/day")
print(f"Gas export rate:       {gas_rate_kg_hr:.1f} kg/hr")
print(f"Oil export rate:       {oil_rate_m3_hr:.2f} m3/hr")
print(f"Oil export rate:       {oil_rate_kg_hr:.1f} kg/hr")
print(f"Total produced water:  {total_water:.2f} m3/hr")
print()
print("COMPRESSOR POWER")
print(f"LP Compressor:         {lp_power_kW:.1f} kW")
print(f"Export Compressor:     {export_power_kW:.1f} kW")
print(f"Total power:           {total_power_kW:.1f} kW")
print("=" * 60)

PRODUCTION SUMMARY
Gas export rate:       0.7823 MSm3/day
Gas export rate:       32862.8 kg/hr
Oil export rate:       21.96 m3/hr
Oil export rate:       14269.8 kg/hr
Total produced water:  3.02 m3/hr

COMPRESSOR POWER
LP Compressor:         58.6 kW
Export Compressor:     2585.3 kW
Total power:           2643.9 kW


## 23.4 Process Conditions Across the Plant

We collect temperature and pressure at each major equipment outlet and
visualize the process conditions across the separation and compression train.

In [5]:
# Collect process conditions
equipment_labels = [
    "Well\nStream", "HP Sep\nGas", "HP Sep\nOil",
    "MP Sep\nGas", "MP Sep\nOil",
    "LP Sep\nGas", "LP Sep\nOil",
    "LP Comp\nOut", "LP Cooler\nOut",
    "Export Comp\nOut", "Export Cooler\nOut"
]

streams = [
    well_stream, hp_sep.getGasOutStream(), hp_sep.getOilOutStream(),
    mp_sep.getGasOutStream(), mp_sep.getOilOutStream(),
    lp_sep.getGasOutStream(), lp_sep.getOilOutStream(),
    lp_compressor.getOutletStream(), lp_cooler.getOutletStream(),
    export_compressor.getOutletStream(), export_cooler.getOutletStream()
]

temperatures_C = [s.getTemperature("C") for s in streams]
pressures_bara = [s.getPressure("bara") for s in streams]
flow_rates_kg_hr = [s.getFlowRate("kg/hr") for s in streams]

# Print table
print(f"{'Equipment':<20} {'T (C)':>8} {'P (bara)':>10} {'Flow (kg/hr)':>14}")
print("-" * 55)
for lbl, t, p, f in zip(equipment_labels, temperatures_C, pressures_bara, flow_rates_kg_hr):
    name = lbl.replace('\n', ' ')
    print(f"{name:<20} {t:>8.1f} {p:>10.1f} {f:>14.1f}")

Equipment               T (C)   P (bara)   Flow (kg/hr)
-------------------------------------------------------
Well Stream              80.0       80.0        50000.0
HP Sep Gas               80.0       80.0        28675.8
HP Sep Oil               80.0       80.0        18456.9
MP Sep Gas               70.4       20.0         2532.2
MP Sep Oil               70.4       20.0        15924.7
LP Sep Gas               57.5        3.0         1654.9
LP Sep Oil               57.5        3.0        14269.8
LP Comp Out             133.2       20.0         1654.9
LP Cooler Out            35.0       20.0         1654.9
Export Comp Out         219.0      150.0        32862.8
Export Cooler Out        40.0      150.0        32862.8


In [6]:
# Plot process conditions
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
x = np.arange(len(equipment_labels))

# Pressures
axes[0].bar(x, pressures_bara, color="steelblue", edgecolor="black")
axes[0].set_ylabel("Pressure (bara)", fontsize=11)
axes[0].set_title("Process Conditions Across the Offshore Platform", fontsize=13, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# Temperatures
axes[1].bar(x, temperatures_C, color="coral", edgecolor="black")
axes[1].set_ylabel("Temperature (°C)", fontsize=11)
axes[1].grid(axis="y", alpha=0.3)

# Flow rates
axes[2].bar(x, flow_rates_kg_hr, color="mediumseagreen", edgecolor="black")
axes[2].set_ylabel("Flow Rate (kg/hr)", fontsize=11)
axes[2].set_xticks(x)
axes[2].set_xticklabels(equipment_labels, fontsize=8, rotation=0)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch23_process_conditions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch23_process_conditions.png")

Figure saved: ../figures/ch23_process_conditions.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_34756\957631895.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 23.5 Power Consumption by Equipment

A bar chart showing the power draw from each piece of rotating equipment
helps identify the dominant energy consumers on the platform.

In [7]:
# Power consumption bar chart
equip_names = ["LP Compressor", "Export Compressor"]
powers_kW = [lp_power_kW, export_power_kW]
colors = ["#4e79a7", "#f28e2b"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(equip_names, powers_kW, color=colors, edgecolor="black", width=0.5)

# Add value labels on bars
for bar, val in zip(bars, powers_kW):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f"{val:.0f} kW", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylabel("Power Consumption (kW)", fontsize=12)
ax.set_title("Power Consumption by Equipment", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

# Add total annotation
ax.annotate(f"Total: {total_power_kW:.0f} kW",
            xy=(0.95, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray"))

plt.tight_layout()
plt.savefig("../figures/ch23_power_consumption.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch23_power_consumption.png")

Figure saved: ../figures/ch23_power_consumption.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_34756\1782951305.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 23.6 Equipment Utilization Summary

We summarize separator gas and liquid loads, and compressor discharge temperatures,
to assess how each piece of equipment is performing relative to typical design limits.

In [8]:
# Separator loads
print("=" * 65)
print("EQUIPMENT UTILIZATION SUMMARY")
print("=" * 65)

separators = [
    ("HP Separator", hp_sep, 80.0),
    ("MP Separator", mp_sep, 20.0),
    ("LP Separator", lp_sep, 3.0),
]

print(f"\n{'Separator':<16} {'P (bara)':>10} {'Gas (kg/hr)':>14} {'Oil (kg/hr)':>14} {'Water (kg/hr)':>14}")
print("-" * 70)
for name, sep, press in separators:
    gas_flow = sep.getGasOutStream().getFlowRate("kg/hr")
    oil_flow = sep.getOilOutStream().getFlowRate("kg/hr")
    water_flow = sep.getWaterOutStream().getFlowRate("kg/hr")
    print(f"{name:<16} {press:>10.1f} {gas_flow:>14.1f} {oil_flow:>14.1f} {water_flow:>14.1f}")

compressors = [
    ("LP Compressor", lp_compressor),
    ("Export Compressor", export_compressor),
]

print(f"\n{'Compressor':<20} {'P_in (bara)':>12} {'P_out (bara)':>12} {'T_out (C)':>10} {'Power (kW)':>12}")
print("-" * 70)
for name, comp in compressors:
    p_in = comp.getInletStream().getPressure("bara")
    p_out = comp.getOutletStream().getPressure("bara")
    t_out = comp.getOutletStream().getTemperature("C")
    power = comp.getPower() / 1000.0
    print(f"{name:<20} {p_in:>12.1f} {p_out:>12.1f} {t_out:>10.1f} {power:>12.1f}")

print("\n" + "=" * 65)

EQUIPMENT UTILIZATION SUMMARY

Separator          P (bara)    Gas (kg/hr)    Oil (kg/hr)  Water (kg/hr)
----------------------------------------------------------------------
HP Separator           80.0        28675.8        18456.9         2867.3
MP Separator           20.0         2532.2        15924.7            0.0
LP Separator            3.0         1654.9        14269.8            0.0

Compressor            P_in (bara) P_out (bara)  T_out (C)   Power (kW)
----------------------------------------------------------------------
LP Compressor                 3.0         20.0      133.2         58.6
Export Compressor            20.0        150.0      219.0       2585.3



## Summary

This case study demonstrated a complete offshore platform simulation using NeqSim,
covering:

- **Three-stage separation** (HP at 80 bara, MP at 20 bara, LP at 3 bara)
- **Gas compression** from LP to export pressure (150 bara)
- **Production accounting** — gas, oil, and water rates
- **Power analysis** — identifying the export compressor as the dominant consumer

This integrated model forms the basis for optimization studies in production
allocation, pressure set-point tuning, and energy efficiency improvements.